# Shimi Poker v2c — קטע s9a (81 פריימים = 5.06 שניות, תנועה אחת)
**הוראות:** Runtime → Change runtime type → T4 GPU → Save. אחר כך Runtime → Run all. הכל אוטומטי.
חזה מחוזק לפי דרישת סאן 12/10. מצב לבוש נעול. סיום בהלבשה תחתונה.

In [ ]:
# 1) בדיקת GPU + התקנות
import torch
assert torch.cuda.is_available(), 'אין GPU! Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
import subprocess
subprocess.run(['pip','install','-q','-U','git+https://github.com/huggingface/diffusers.git','transformers>=4.51','accelerate','ftfy','protobuf','imageio[ffmpeg]','safetensors','requests'], check=True)
print('התקנות הושלמו')

In [ ]:
# 2) טעינת Wan 2.2 TI2V-5B — מקודד טקסט bf16 על CPU (ללא NaN), טרנספורמר fp16 על GPU
import torch, time
from diffusers import WanPipeline, AutoencoderKLWan, WanTransformer3DModel
from transformers import AutoTokenizer, UMT5EncoderModel
t0 = time.time()
MID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
text_encoder = UMT5EncoderModel.from_pretrained(MID, subfolder='text_encoder', torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MID, subfolder='tokenizer')
vae = AutoencoderKLWan.from_pretrained(MID, subfolder='vae', torch_dtype=torch.float32)
transformer = WanTransformer3DModel.from_pretrained(MID, subfolder='transformer', torch_dtype=torch.float16)
pipe = WanPipeline.from_pretrained(MID, text_encoder=text_encoder, tokenizer=tokenizer, vae=vae, transformer=transformer)
pipe.enable_model_cpu_offload()
if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
print('נטען ב', round(time.time()-t0,1), 'שניות')

In [ ]:
PROMPT = 'A pale goth woman, pale porcelain skin with freckles, sharp black winged eyeliner, dark plum lipstick, long straight jet-black hair with straight blunt bangs and one deep-purple streak, silver eyebrow piercing. LOCKED BODY: pronounced hourglass figure, very large full heavy natural bust, deep cleavage, very narrow small waist, wide full hips, soft rounded thighs, soft natural belly with realistic fat distribution, average height, same proportions in every frame. She wears an unpadded soft-cup black embroidered lace bra with fine floral thread pattern, scalloped edges and thin elastic straps, an opaque embroidered fabric that follows her natural shape without compressing or lifting, and matching soft high-waist black briefs. She bends forward deeply at the waist, forearms resting on her knees, glancing back over her shoulder with a teasing smile. Real gravity physics: her soft heavy bust hangs downward and shifts with the bend, the soft flesh of her belly folds gently as she leans, her hips and thighs keep their full natural size under the soft fabric. Underground poker room, green felt table with chips and cards, warm brass lamp light overhead, her friend laughs at the table. Handheld cinematic camera, natural motion, film grain, photorealistic, realistic skin texture with visible pores, natural soft tissue movement under gravity, no warping or stretching, stable face, anatomically correct hands with exactly five fingers on each hand.'
NEG = ('bright tones, overexposed, static, blurred details, subtitles, style of painting, paintings, images, static, overall gray, worst quality, low quality, ugly, incomplete, extra fingers, poorly drawn hands, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, cartoon, anime, illustration, cgi, 3d render, rigid plastic body, frozen soft tissue, defying gravity, compressed flattened bust')

FRAMES = 81
STEPS = 20
SEED = 711
print(FRAMES, STEPS, SEED)


In [ ]:
# 4) רנדור קטע s9a (t2v) + וידוא שאין שחור + שמירה
import torch, numpy as np, imageio, time, os
from PIL import Image
t0 = time.time()
out = pipe(
    prompt=PROMPT, negative_prompt=NEG,
    height=480, width=832, num_frames=FRAMES,
    num_inference_steps=STEPS, guidance_scale=5.0, max_sequence_length=512,
    generator=torch.Generator('cpu').manual_seed(SEED),
).frames[0]
a = np.asarray(out[0].convert('RGB')).astype('float32')
print('ממוצע בהירות פריים ראשון:', round(float(a.mean()),1))
assert a.mean() > 5, 'פלט שחור'
imageio.mimwrite('physics_bend_soft.mp4', [np.asarray(f.convert('RGB')) for f in out], fps=16, quality=9)
out[0].save('physics_bend_soft_first.png')
out[-1].save('physics_bend_soft_last.png')
print('סיום ב', round((time.time()-t0)/60,1), 'דקות | גודל:', round(os.path.getsize('physics_bend_soft.mp4')/1e6,1), 'MB')

In [ ]:
# 5) העלאה ל-catbox + דיווח לגשר שימי פיתוח
import requests
def catbox(path):
    r = requests.post('https://catbox.moe/user/api.php',
        data={'reqtype':'fileupload'}, files={'fileToUpload': open(path,'rb')}, timeout=600)
    r.raise_for_status(); return r.text
url_mp4 = catbox('s9a.mp4')
url_first = catbox('s9a_first.png')
url_last = catbox('s9a_last.png')
print('MP4:', url_mp4)
print('FIRST:', url_first)
print('LAST:', url_last)
try:
    requests.post('https://app.base44.com/api/apps/6a901f55a4d9a9b76a095a77/backend_functions/pokerColabRelay',
        json={'action':'report','scene':'s9a','url':url_mp4,'first':url_first,'last':url_last,
              'source':'colab','frames':81}, timeout=30)
    print('דווח לגשר')
except Exception as e:
    print('דיווח לגשר נכשל (הקישורים למעלה תקפים):', e)